In [4]:
import os
import gc
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree

In [5]:
# ==============================================================================
# 1. LOAD NATIONAL RZLT CENTROID COORDINATES (MEMORY-EFFICIENT)
# ==============================================================================
national_path = "../Datasets/RZLT-Dataset.geojson"
if not os.path.exists(national_path):
    raise FileNotFoundError(f"Could not find '{national_path}'. Update the path to your full 260k RZLT file.")

print("Loading National RZLT layer to extract physical coordinates...")
# Read only geometry (and PARCEL_ID if needed) to keep RAM usage minimal
gdf_nat = gpd.read_file(national_path, columns=['PARCEL_ID', 'geometry'])

if gdf_nat.crs != "EPSG:2157":
    gdf_nat = gdf_nat.to_crs(epsg=2157)

nat_coords = np.column_stack([gdf_nat.geometry.centroid.x, gdf_nat.geometry.centroid.y])
print(f"Extracted {len(nat_coords):,} national parcel centroids in EPSG:2157.")

# Build national spatial k-d tree and free the heavy 260k GeoDataFrame immediately
nat_tree = cKDTree(nat_coords)
del gdf_nat, nat_coords
gc.collect()
print("National cKDTree built and raw GeoDataFrame cleared from RAM.")

Loading National RZLT layer to extract physical coordinates...
Extracted 260,463 national parcel centroids in EPSG:2157.
National cKDTree built and raw GeoDataFrame cleared from RAM.


In [6]:
# ==============================================================================
# 2. LOAD GTFS-ENRICHED SAMPLE & COMPUTE TRUE NATIONAL DENSITY
# ==============================================================================
sample_path = os.path.join("../Datasets", "RZLT_Samples", "RZLT_AllCodes_10to1_WithGTFS.geojson")
if not os.path.exists(sample_path):
    sample_path = "RZLT_AllCodes_10to1_WithGTFS.geojson"

gdf_sample = gpd.read_file(sample_path)
if gdf_sample.crs != "EPSG:2157":
    gdf_sample = gdf_sample.to_crs(epsg=2157)

sample_coords = np.column_stack([gdf_sample['centroid_x'], gdf_sample['centroid_y']])

# Query true national neighbor counts within 500m and 1000m (subtract 1 to exclude self)
neighbors_500m = nat_tree.query_ball_point(sample_coords, r=500.0)
neighbors_1000m = nat_tree.query_ball_point(sample_coords, r=1000.0)

gdf_sample['TRUE_PARCELS_WITHIN_500M'] = [max(0, len(idx) - 1) for idx in neighbors_500m]
gdf_sample['TRUE_PARCELS_WITHIN_1000M'] = [max(0, len(idx) - 1) for idx in neighbors_1000m]
gdf_sample['LOG_TRUE_PARCELS_500M'] = np.log1p(gdf_sample['TRUE_PARCELS_WITHIN_500M'])

In [7]:
# ==============================================================================
# 3. AUDIT BEFORE vs. AFTER & SAVE UPDATED DATASET
# ==============================================================================
print("\n=== True National Neighbor Density Summary ===")
density_cols = ['TRUE_PARCELS_WITHIN_500M', 'TRUE_PARCELS_WITHIN_1000M']
print(gdf_sample[density_cols].describe().round(1))

print("\n=== Median True Density by Decision Status ===")
print(gdf_sample.groupby('DECISION_STATUS')[density_cols].median().round(1))

# Also ensure geometry shape features are persisted in the saved file
area_m2 = gdf_sample.geometry.area
perim_m = gdf_sample.geometry.length
gdf_sample['SHAPE_COMPACTNESS'] = (4 * np.pi * area_m2) / np.maximum(perim_m ** 2, 1e-6)
gdf_sample['PERIM_TO_AREA'] = perim_m / np.maximum(area_m2, 1e-6)

output_path = os.path.join("../Datasets", "RZLT_Samples", "RZLT_AllCodes_10to1_FullFeatures.geojson")
gdf_sample.to_file(output_path, driver="GeoJSON")
print(f"\nSaved complete feature-enriched dataset -> {output_path}")


=== True National Neighbor Density Summary ===
       TRUE_PARCELS_WITHIN_500M  TRUE_PARCELS_WITHIN_1000M
count                    6303.0                     6303.0
mean                      669.5                     2059.1
std                       503.9                     1614.5
min                         0.0                        2.0
25%                       200.0                      509.0
50%                       657.0                     1948.0
75%                      1029.5                     3226.0
max                      2709.0                     7253.0

=== Median True Density by Decision Status ===
                 TRUE_PARCELS_WITHIN_500M  TRUE_PARCELS_WITHIN_1000M
DECISION_STATUS                                                     
Excluded                             37.0                       96.0
Included_Liable                     717.0                     2150.5

Saved complete feature-enriched dataset -> ../Datasets\RZLT_Samples\RZLT_AllCodes_10to1_FullFeat